In [0]:
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

def path_exists(p):
    try:
        dbutils.fs.ls(p)
        return True
    except Exception as e:
        if "FileNotFoundException" in str(e) or "does not exist" in str(e).lower() or "not found" in str(e).lower():
            return False
        raise

if not path_exists(path):
    msg = f"No new files received. Path does not exist: {path}"
    print(msg)
    dbutils.notebook.exit(msg)

In [0]:
"""
if source_file_format == "csv":
    landing_df =spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/")
else:
    landing_df = spark.read.format("parquet").load(f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/")

"""
if source_file_format == "csv":
    landing_df = spark.read.csv(path, header=True, inferSchema=True)
else:
    landing_df = spark.read.parquet(path)


In [0]:
%skip
display(df)

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_TimeStamp", F.lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", F.current_timestamp())

In [0]:
%skip
display(updated_df)


In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")

In [0]:
%skip
%sql
select * from adb_caresync.bronze.hospitals 